In [1]:
import glob
import os 
import yaml
import cv2

import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt 
import seaborn as sns

from aniposelib.cameras import Camera, CameraGroup, FisheyeCamera
from einops import rearrange

from posetail_preprocessing.utils import io
from posetail_preprocessing.utils.calibration import disassemble_extrinsics, assemble_extrinsics

%load_ext autoreload
%autoreload 2

In [2]:
def format_scheme(scheme, keypoint_names):

    new_scheme = [] 
    kpt_to_ix = dict(zip(keypoint_names, range(len(keypoint_names))))

    for kpt1, kpt2 in scheme: 
        new_scheme.append([kpt_to_ix[kpt1], kpt_to_ix[kpt2]])

    return new_scheme

In [ ]:
data_path = '/groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/'
splits = ['train', 'val', 'test']
scheme_prefix = '/home/ruppk2@hhmi.org/posetail-preprocessing/posetail_preprocessing/schemes'

datasets = ['3dpop', 'allen-mouse', 'anipose-fly', 'johnson-mouse', 'cmupanoptic', 'rat7m', 'pair-r24m']

for dataset in io.get_dirs(data_path): 

    print(dataset)
    if dataset not in datasets:
        continue

    print(f'processing {dataset}')
    scheme_path = os.path.join(scheme_prefix, f'scheme_{dataset}.toml')
    if not os.path.exists(scheme_path): 
        continue
    
    # load the metadata config
    scheme = io.load_toml(scheme_path)['scheme']
    print(scheme)

    for split in splits: 
        
        dataset_path = os.path.join(data_path, dataset, split)

        for session in io.get_dirs(dataset_path): 
            session_path = os.path.join(dataset_path, session)

            for trial in io.get_dirs(session_path):
                # get paths to metadata, 3d pose, and images
                trial_path = os.path.join(session_path, trial)

                metadata_path = os.path.join(trial_path, 'metadata.yaml')
                assert os.path.exists(metadata_path)
                cam_metadata = io.load_yaml(metadata_path)

                pose_path = os.path.join(trial_path, 'pose3d.npz')
                assert os.path.exists(pose_path) 
                data = np.load(pose_path, allow_pickle = True)

                data_dict = {k: data[k] for k in data.files}
                data_dict['scheme'] = np.array(scheme)
                print(data_dict['keypoints'])
                scheme_ixs = format_scheme(scheme, data_dict['keypoints'])
                # data_dict['scheme_ixs'] = np.array(scheme_ixs, dtype = np.int32)
                    
                # print(list(cam_metadata.keys()))
                print(data_dict.keys())
                
                # io.save_yaml(data = cam_metadata, outpath = trial_path, 
                #     fname = 'metadata.yaml')
                print(scheme)
                print(pose_path)
                # np.savez(pose_path, **data_dict)


In [ ]:
# remove -1 subject from cmupanoptic (all nans)

# data_path = '/groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/'
# splits = ['train', 'val', 'test']
# splits = ['val']
# datasets = ['cmupanoptic']

# for dataset in io.get_dirs(data_path): 

#     if dataset not in datasets:
#         continue
    
#     for split in splits: 
        
#         dataset_path = os.path.join(data_path, dataset, split)

#         for session in io.get_dirs(dataset_path): 
#             session_path = os.path.join(dataset_path, session)

#             for trial in io.get_dirs(session_path):

#                 trial_path = os.path.join(session_path, trial)
#                 pose_path = os.path.join(trial_path, 'pose3d.npz')
                
#                 assert os.path.exists(pose_path) 
#                 data = np.load(pose_path, allow_pickle = True)

#                 data_dict = {k: data[k] for k in data.files}
#                 print(trial_path)
#                 print(data_dict.keys())

#                 ids = np.array(data_dict['ids'])
#                 print(ids)

#                 # update the pose file
#                 # data_dict['pose'] = updated_pose
#                 # data_dict['vis'] = updated_vis
#                 # data_dict['subject_ids'] = updated_subject_ids
#                 # np.savez(pose_path, **data_dict)


# ids.dtype

/groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/cmupanoptic/val/160906_pizza1/trial
dict_keys(['pose', 'vis', 'keypoints', 'ids'])
{0, 1, 2, 3, 4, 5, 6, -1}


dtype('O')

In [ ]:
# remove cam 4 from pair r24m 
import shutil 
from pathlib import Path

data_path = '/groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/pair-r24m'
splits = ['test']
# splits = ['train', 'val', 'test']
cam_to_remove = 'Camera4'

for split in splits: 
    
    dataset_path = os.path.join(data_path, split)

    for session in io.get_dirs(dataset_path): 
        session_path = os.path.join(dataset_path, session)

        for trial in io.get_dirs(session_path):
            # get paths to metadata, 3d pose, and images
            trial_path = os.path.join(session_path, trial)

            metadata_path = os.path.join(trial_path, 'metadata.yaml')
            cam_metadata = io.load_yaml(metadata_path)
            print(list(cam_metadata.keys()))

            # remove camera from the metadata and save updated metadata
            keys = ['intrinsic_matrices', 'extrinsic_matrices', 'distortion_matrices', 
                    'camera_heights', 'camera_widths']
            for k in keys: 
                if cam_to_remove in cam_metadata[k]: 
                    cam_metadata[k].pop(cam_to_remove)

            n_cams = cam_metadata['num_cameras']
            cam_metadata['num_cameras'] = n_cams - 1
            io.save_yaml(data = cam_metadata, outpath = trial_path, fname = 'metadata.yaml')

            # remove camera videos for inference
            if split == 'test': 
                vid_path = Path(os.path.join(trial_path, 'vid', f'{cam_to_remove}.mp4'))
                if vid_path.is_symlink():
                    print(f'removing symlink for {vid_path}')
                    # vid_path.unlink()

            # remove camera images for train and val
            else: 
                img_path = os.path.join(trial_path, 'img', cam_to_remove)
                if os.path.exists(img_path):
                    print(f'removing {img_path}')
                    # shutil.rmtree(img_path)

['camera_heights', 'camera_widths', 'distortion_matrices', 'extrinsic_matrices', 'fps', 'intrinsic_matrices', 'num_cameras', 'num_frames']
removing symlink for /groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/pair-r24m/test/20210218_Recording_SR10_SR11m_social_wvid/0/vid/Camera4.mp4
['camera_heights', 'camera_widths', 'distortion_matrices', 'extrinsic_matrices', 'fps', 'intrinsic_matrices', 'num_cameras', 'num_frames']
removing symlink for /groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/pair-r24m/test/20210218_Recording_SR10_SR11m_social_wvid/101500/vid/Camera4.mp4
['camera_heights', 'camera_widths', 'distortion_matrices', 'extrinsic_matrices', 'fps', 'intrinsic_matrices', 'num_cameras', 'num_frames']
removing symlink for /groups/karashchuk/karashchuklab/animal-datasets-processed/posetail-finetuning/pair-r24m/test/20210218_Recording_SR10_SR11m_social_wvid/10500/vid/Camera4.mp4
['camera_heights', 'camera_widths', 'distortion_m

In [18]:
# check cmupanoptic output 

data_path = '/home/ruppk2@hhmi.org/dataset_scripts/test/cmupanoptic2/train/160224_haggling1/trial/pose3d.npz'
data = np.load(data_path)
data['pose'].shape
list(data.keys())
scheme = data['scheme']


array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])